# Feature Engineering
EDA에서 생성한 `demand_filled`를 기반으로 ARIMA / XGBoost / LSTM에 필요한 feature를 생성합니다.

## 1. 라이브러리 및 EDA 결과물 불러오기

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

# -----------------------------------------------
# EDA 노트북에서 이어받는 핵심 변수
# 별도 실행 시 아래 블록을 그대로 복붙해서 재생성하세요
# -----------------------------------------------
ANALYSIS_START  = '2017-01-01'
ANALYSIS_END    = '2018-08-31'
TRAIN_END       = '2018-06-30'
TEST_START      = '2018-07-01'
TOP_N           = 10

## 2. EDA 결과물 재생성 (독립 실행용)

> EDA 노트북과 이어서 실행하는 경우 이 셀은 건너뛰어도 됩니다.  
> 이 노트북을 단독으로 실행할 때는 반드시 실행하세요.

In [ ]:
path = "../data/"

orders      = pd.read_csv(path + "olist_orders_dataset.csv")
order_items = pd.read_csv(path + "olist_order_items_dataset.csv")
products    = pd.read_csv(path + "olist_products_dataset.csv")

orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_date'] = orders['order_purchase_timestamp'].dt.normalize()

orders = orders[
    (orders['order_date'] >= ANALYSIS_START) &
    (orders['order_date'] <= ANALYSIS_END)
]

merged = pd.merge(
    orders[['order_id', 'order_date']],
    order_items[['order_id', 'product_id', 'freight_value']],
    on='order_id', how='inner'
)
merged = pd.merge(
    merged,
    products[['product_id', 'product_category_name']],
    on='product_id', how='left'
)
merged = merged.dropna(subset=['product_category_name'])

demand_df = (
    merged
    .groupby(['order_date', 'product_category_name'])
    .size()
    .reset_index(name='demand')
)

TOP_CATEGORY_LIST = (
    demand_df
    .groupby('product_category_name')['demand'].sum()
    .sort_values(ascending=False)
    .head(TOP_N)
    .index.tolist()
)

full_date_range = pd.date_range(start=ANALYSIS_START, end=ANALYSIS_END, freq='D')
filled_frames = []
for cat in TOP_CATEGORY_LIST:
    cat_df = (
        demand_df[demand_df['product_category_name'] == cat]
        .set_index('order_date')[['demand']]
        .reindex(full_date_range, fill_value=0)
    )
    cat_df.index.name = 'order_date'
    cat_df['product_category_name'] = cat
    filled_frames.append(cat_df.reset_index())

demand_filled = pd.concat(filled_frames, ignore_index=True)

print("demand_filled shape:", demand_filled.shape)
print("categories:", TOP_CATEGORY_LIST)

## 3. Feature Engineering 함수 정의

카테고리별로 독립적인 시계열로 처리해야 하므로,  
반드시 **카테고리 내부에서만** shift/rolling을 계산해야 합니다.  
전체 df에서 그대로 shift()를 쓰면 카테고리 경계에서 값이 오염됩니다.

In [ ]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    단일 카테고리 시계열 df에 feature를 추가합니다.
    입력 df는 반드시 order_date 기준 오름차순 정렬 상태여야 합니다.

    생성 feature:
        lag_1          : 전일 수요
        lag_7          : 7일 전 수요 (동일 요일 효과 포착)
        lag_14         : 14일 전 수요 (2주 전 패턴)
        rolling_mean_7 : 7일 이동평균 (단기 추세)
        rolling_std_7  : 7일 이동표준편차 (단기 변동성)
        rolling_mean_14: 14일 이동평균 (중기 추세)
        weekday        : 요일 (0=월 ~ 6=일)
        is_weekend     : 주말 여부
        month          : 월
        day_of_year    : 연중 일수 (계절성 포착)
    """
    df = df.copy().sort_values('order_date').reset_index(drop=True)

    # --- lag features ---
    df['lag_1']  = df['demand'].shift(1)
    df['lag_7']  = df['demand'].shift(7)
    df['lag_14'] = df['demand'].shift(14)

    # --- rolling features (min_periods=1로 초반 결측 최소화) ---
    df['rolling_mean_7']  = df['demand'].shift(1).rolling(window=7,  min_periods=1).mean()
    df['rolling_std_7']   = df['demand'].shift(1).rolling(window=7,  min_periods=2).std().fillna(0)
    df['rolling_mean_14'] = df['demand'].shift(1).rolling(window=14, min_periods=1).mean()

    # --- calendar features ---
    df['weekday']     = df['order_date'].dt.weekday          # 0=월, 6=일
    df['is_weekend']  = (df['weekday'] >= 5).astype(int)
    df['month']       = df['order_date'].dt.month
    df['day_of_year'] = df['order_date'].dt.dayofyear

    return df


print("add_features 함수 정의 완료")
print("생성될 feature:", [
    'lag_1', 'lag_7', 'lag_14',
    'rolling_mean_7', 'rolling_std_7', 'rolling_mean_14',
    'weekday', 'is_weekend', 'month', 'day_of_year'
])

## 4. 카테고리별 Feature Engineering 실행

In [ ]:
fe_frames = []

for cat in TOP_CATEGORY_LIST:
    cat_df = demand_filled[demand_filled['product_category_name'] == cat].copy()
    cat_df = add_features(cat_df)
    fe_frames.append(cat_df)

fe_df = pd.concat(fe_frames, ignore_index=True)

print("Feature Engineering 완료")
print("shape:", fe_df.shape)
print("\n컬럼 목록:")
print(fe_df.columns.tolist())
print("\n샘플 (첫 카테고리, 상위 20행):")
fe_df[fe_df['product_category_name'] == TOP_CATEGORY_LIST[0]].head(20)

## 5. 결측치 확인

lag/rolling feature는 초반 N일이 NaN입니다.  
- lag_1  → 1행 NaN
- lag_7  → 7행 NaN  
- lag_14 → 14행 NaN  

ARIMA는 원본 시계열을 그대로 쓰므로 무관합니다.  
XGBoost / LSTM은 NaN 행을 제거하고 사용합니다.

In [ ]:
print("전체 결측치 현황:")
print(fe_df.isnull().sum())

# lag_14 기준으로 NaN 제거 (가장 긴 lag)
fe_df_clean = fe_df.dropna(subset=['lag_14']).copy()

print(f"\nNaN 제거 전: {len(fe_df)}행")
print(f"NaN 제거 후: {len(fe_df_clean)}행")
print(f"제거 비율: {(len(fe_df) - len(fe_df_clean)) / len(fe_df) * 100:.1f}%")

## 6. Train / Test split

시간 순서를 반드시 지켜야 합니다.  
`random_state` 기반 split을 사용하면 미래 데이터가 train에 포함되는 data leakage가 발생합니다.

In [ ]:
# --- 전체 데이터 (ARIMA용: feature 없이 원본 시계열) ---
train_raw = demand_filled[demand_filled['order_date'] <= TRAIN_END].copy()
test_raw  = demand_filled[demand_filled['order_date'] >= TEST_START].copy()

# --- Feature 포함 데이터 (XGBoost / LSTM용) ---
train_fe = fe_df_clean[fe_df_clean['order_date'] <= TRAIN_END].copy()
test_fe  = fe_df_clean[fe_df_clean['order_date'] >= TEST_START].copy()

print("=" * 40)
print("ARIMA용 (원본 시계열)")
print(f"  train_raw : {train_raw.shape}")
print(f"  test_raw  : {test_raw.shape}")
print()
print("XGBoost / LSTM용 (feature 포함)")
print(f"  train_fe  : {train_fe.shape}")
print(f"  test_fe   : {test_fe.shape}")
print()
print("train 기간:", train_raw['order_date'].min().date(), "~", train_raw['order_date'].max().date())
print("test  기간:", test_raw['order_date'].min().date(),  "~", test_raw['order_date'].max().date())

## 7. XGBoost용 feature / target 분리

In [ ]:
FEATURE_COLS = [
    'lag_1', 'lag_7', 'lag_14',
    'rolling_mean_7', 'rolling_std_7', 'rolling_mean_14',
    'weekday', 'is_weekend', 'month', 'day_of_year'
]
TARGET_COL = 'demand'

X_train = train_fe[FEATURE_COLS]
y_train = train_fe[TARGET_COL]

X_test  = test_fe[FEATURE_COLS]
y_test  = test_fe[TARGET_COL]

print("XGBoost feature / target 분리 완료")
print(f"  X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"  X_test : {X_test.shape},  y_test : {y_test.shape}")
print()
print("Feature 통계 (train):")
X_train.describe().round(2)

## 8. LSTM용 시퀀스 데이터 생성

LSTM은 `(samples, timesteps, features)` 형태의 3D 배열을 입력으로 받습니다.  
window_size = 14로 설정하면 과거 14일치 데이터로 다음날 수요를 예측합니다.

> 주의: 카테고리 경계를 넘어서 시퀀스를 만들면 안 됩니다.  
> 반드시 카테고리별로 분리해서 시퀀스를 생성합니다.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

WINDOW_SIZE = 14  # 과거 N일치로 다음날 예측


def make_sequences(series: np.ndarray, window: int):
    """
    1D 시계열 배열을 (X, y) 시퀀스 쌍으로 변환합니다.

    Args:
        series : 1D numpy array (단일 카테고리 수요)
        window : 입력 시퀀스 길이

    Returns:
        X : shape (n_samples, window, 1)
        y : shape (n_samples,)
    """
    X, y = [], []
    for i in range(len(series) - window):
        X.append(series[i : i + window])
        y.append(series[i + window])
    return np.array(X).reshape(-1, window, 1), np.array(y)


lstm_data = {}  # {category: {'X_train', 'y_train', 'X_test', 'y_test', 'scaler'}}

for cat in TOP_CATEGORY_LIST:
    cat_train = train_raw[train_raw['product_category_name'] == cat]['demand'].values.astype(float)
    cat_test  = test_raw[test_raw['product_category_name']   == cat]['demand'].values.astype(float)

    # 카테고리별 스케일러 (train 기준으로만 fit)
    scaler = MinMaxScaler(feature_range=(0, 1))
    cat_train_scaled = scaler.fit_transform(cat_train.reshape(-1, 1)).flatten()

    # test는 train scaler로 transform (leakage 방지)
    cat_test_scaled  = scaler.transform(cat_test.reshape(-1, 1)).flatten()

    # 시퀀스 생성
    # test 시퀀스 생성 시 train 마지막 window개를 앞에 붙여서 연속성 확보
    combined_scaled = np.concatenate([cat_train_scaled[-WINDOW_SIZE:], cat_test_scaled])

    X_tr, y_tr = make_sequences(cat_train_scaled, WINDOW_SIZE)
    X_te, y_te = make_sequences(combined_scaled,  WINDOW_SIZE)

    lstm_data[cat] = {
        'X_train': X_tr,
        'y_train': y_tr,
        'X_test' : X_te,
        'y_test' : y_te,
        'scaler' : scaler,
    }

# 결과 확인 (첫 카테고리)
sample_cat = TOP_CATEGORY_LIST[0]
d = lstm_data[sample_cat]
print(f"LSTM 시퀀스 생성 완료 (window={WINDOW_SIZE})")
print(f"\n카테고리: {sample_cat}")
print(f"  X_train : {d['X_train'].shape}  ← (samples, timesteps, features)")
print(f"  y_train : {d['y_train'].shape}")
print(f"  X_test  : {d['X_test'].shape}")
print(f"  y_test  : {d['y_test'].shape}")

## 9. Feature 분포 시각화

생성된 feature가 의도한 패턴을 잘 잡고 있는지 확인합니다.

In [ ]:
target_cat = TOP_CATEGORY_LIST[0]
cat_fe = fe_df_clean[fe_df_clean['product_category_name'] == target_cat].copy()

fig, axes = plt.subplots(3, 2, figsize=(14, 10))

# --- demand + rolling_mean_7 ---
ax = axes[0, 0]
ax.plot(cat_fe['order_date'], cat_fe['demand'], alpha=0.5, label='demand', linewidth=0.8)
ax.plot(cat_fe['order_date'], cat_fe['rolling_mean_7'], label='rolling_mean_7', linewidth=1.2)
ax.set_title(f"{target_cat} — demand vs rolling_mean_7")
ax.legend(fontsize=8)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%y-%m'))
ax.tick_params(axis='x', rotation=45)

# --- lag_7 scatter ---
ax = axes[0, 1]
ax.scatter(cat_fe['lag_7'], cat_fe['demand'], alpha=0.3, s=8)
ax.set_xlabel('lag_7')
ax.set_ylabel('demand')
ax.set_title('demand vs lag_7 (자기상관 확인)')
corr = cat_fe[['demand', 'lag_7']].corr().iloc[0, 1]
ax.text(0.05, 0.92, f'corr = {corr:.3f}', transform=ax.transAxes, fontsize=9)

# --- 요일별 평균 수요 ---
ax = axes[1, 0]
weekday_avg = cat_fe.groupby('weekday')['demand'].mean()
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
ax.bar(days, weekday_avg.values)
ax.set_title('요일별 평균 수요 (weekday feature 유효성)')
ax.set_ylabel('avg demand')

# --- 월별 평균 수요 ---
ax = axes[1, 1]
month_avg = cat_fe.groupby('month')['demand'].mean()
ax.bar(month_avg.index, month_avg.values)
ax.set_title('월별 평균 수요 (month feature 유효성)')
ax.set_xlabel('month')
ax.set_ylabel('avg demand')

# --- rolling_std_7 (변동성 추이) ---
ax = axes[2, 0]
ax.plot(cat_fe['order_date'], cat_fe['rolling_std_7'], linewidth=0.8, color='orange')
ax.set_title('rolling_std_7 — 단기 변동성 추이')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%y-%m'))
ax.tick_params(axis='x', rotation=45)

# --- feature 상관관계 ---
ax = axes[2, 1]
corr_cols = ['demand', 'lag_1', 'lag_7', 'lag_14',
             'rolling_mean_7', 'rolling_mean_14', 'weekday', 'month']
corr_matrix = cat_fe[corr_cols].corr()['demand'].drop('demand').sort_values()
colors = ['#d85a30' if v < 0 else '#1d9e75' for v in corr_matrix.values]
ax.barh(corr_matrix.index, corr_matrix.values, color=colors)
ax.axvline(0, color='gray', linewidth=0.5)
ax.set_title('demand와의 feature 상관계수')
ax.set_xlabel('correlation')

plt.suptitle(f"Feature 분포 검증 — {target_cat}", y=1.01, fontsize=12)
plt.tight_layout()
plt.show()

## 10. 최종 데이터 상태 확인

다음 단계(모델링)로 넘어가기 전 모든 데이터 객체를 정리합니다.

In [ ]:
print("=" * 55)
print("모델링 진입 전 최종 데이터 체크")
print("=" * 55)
print()
print("[ 공통 설정 ]")
print(f"  분석 기간     : {ANALYSIS_START} ~ {ANALYSIS_END}")
print(f"  Train 기간    : {ANALYSIS_START} ~ {TRAIN_END}")
print(f"  Test 기간     : {TEST_START} ~ {ANALYSIS_END}")
print(f"  대상 카테고리 : {len(TOP_CATEGORY_LIST)}개")
print(f"  LSTM window   : {WINDOW_SIZE}일")
print()
print("[ ARIMA용 ]")
print(f"  train_raw : {train_raw.shape}")
print(f"  test_raw  : {test_raw.shape}")
print()
print("[ XGBoost용 ]")
print(f"  X_train   : {X_train.shape}")
print(f"  X_test    : {X_test.shape}")
print(f"  features  : {FEATURE_COLS}")
print()
print("[ LSTM용 ]")
for cat in TOP_CATEGORY_LIST:
    d = lstm_data[cat]
    print(f"  {cat[:30]:<30}  "
          f"X_train={d['X_train'].shape}  X_test={d['X_test'].shape}")
print()
print("다음 단계: 03_modeling.ipynb")